In [ ]:
import cv2
import mediapipe as mp
from keras.models import load_model
import numpy as np

# Cài đặt Mediapipe (Chỉ nhận 1 tay cho gọn)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

# Load model (LƯU Ý: Sửa lại tên file .h5 cho đúng với mô hình của bước này)
model = load_model("E:\\ML_project\\models\\model_kbb3.h5")
danh_sach_chu = ["bao", "bua", "keo"]

In [ ]:
def khoang_cach_toa_do(hand_lm):
    danh_sach_toa_do = []
    for lm in hand_lm.landmark:
        danh_sach_toa_do.append([lm.x, lm.y, lm.z])
    
    toa_do = np.array(danh_sach_toa_do)
    co_tay = toa_do[0]
    chieu_dai = np.linalg.norm(toa_do[9] - co_tay) + 1e-6
    
    chuan_hoa = (toa_do - co_tay) / chieu_dai
    return chuan_hoa.flatten()

In [ ]:
cam = cv2.VideoCapture(0)

while True:
    ret, frame = cam.read()    
    if not ret: break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)    
    h_frames, w_frames, c = frame.shape

    if results.multi_hand_landmarks:
        for hand_lm in results.multi_hand_landmarks:
            # 1. Vẽ khung xương
            mp_draw.draw_landmarks(frame, hand_lm, mp_hands.HAND_CONNECTIONS)
            
            # 2. Xử lý tọa độ và Dự đoán
            du_lieu_tay = khoang_cach_toa_do(hand_lm)
            du_lieu = np.array([du_lieu_tay])
            
            du_doan = model.predict(du_lieu, verbose=0)   
            ket_qua_idx = np.argmax(du_doan)    
            phan_tram = np.max(du_doan) 
            ket_qua_chu = danh_sach_chu[ket_qua_idx]
            
            # 3. In kết quả lên màn hình (neo ở vị trí cổ tay)
            x_co_tay = int(hand_lm.landmark[0].x * w_frames)
            y_co_tay = int(hand_lm.landmark[0].y * h_frames)
            
            text = f"{ket_qua_chu.upper()}: {phan_tram*100:.1f}%"
            cv2.putText(frame, text, (x_co_tay - 20, y_co_tay - 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    cv2.imshow("Test Keo Bua Bao", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('l'):
        break

cam.release()
cv2.destroyAllWindows()